# 3. Entrenamiento fold a fold y evaluación

Con la arquitectura seleccionada para cada (lag, H) entrenamos el MLP multisalida fold a fold con las tres estrategias de `timeseries-cv` (Group K-Fold, Forward Chaining y K-Fold) y calculamos, en USD y sobre validación, MAPE, MAE, RMSE y MSE por paso de horizonte y agregados, junto con el p-valor BDS de los residuos de h = 1. Comparamos con dos referencias: `MLPRegressor` de scikit-learn y el pronóstico ingenuo (último precio observado repetido H días). El modelo final de cada configuración, reentrenado con toda la serie previa al hold-out, se evalúa aparte en el hold-out.

Este notebook no entrena: lee los resultados de las fases F5 y F6 de `src/train_t2.py`.

In [ ]:
# Rutas: en Colab se usa la carpeta de Drive; en el repositorio, la raíz del proyecto.
import os, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DRIVE = Path("/content/drive/MyDrive/tarea2_series")
if DRIVE.exists():
    BASE, FIGS = DRIVE, DRIVE / "figs"
else:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    FIGS = BASE / "notebooks" / "figs"
RESULTS = BASE / os.environ.get("T2_RESULTS_DIR", "results")
FIGS.mkdir(parents=True, exist_ok=True)

LAGS, HS, ESTRATEGIAS = [7, 14, 21, 28], [7, 14, 21, 28], ["gk", "fc", "kf"]
NOMBRE = {"gk": "Group K-Fold", "fc": "Forward Chaining", "kf": "K-Fold"}
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # orden categorico fijo
COL_EST = dict(zip(ESTRATEGIAS, COL))
COL_LAG = dict(zip(LAGS, COL))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.prop_cycle": plt.cycler(color=COL), "lines.linewidth": 1.6,
                     "legend.frameon": False, "axes.titlesize": 11, "axes.labelsize": 10})

df = pd.read_csv(BASE / "data" / "btc_1d.csv", parse_dates=["date"])
y = df["close"].to_numpy(float)
DATES = df["date"].to_numpy()
N_HOLD = max(int(len(y) * 0.10), 56)
N_CV = len(y) - N_HOLD


def guardar(fig, nombre):
    fig.savefig(FIGS / f"{nombre}.png", bbox_inches="tight", dpi=150)

In [ ]:
def cargar_registros():
    R = {}
    for f in sorted(glob.glob(str(RESULTS / "raw" / "*.jsonl"))):
        for line in open(f):
            if line.strip():
                r = json.loads(line); R[r["id"]] = r
    return list(R.values())

REG = cargar_registros()
print(f"{len(REG)} registros en {RESULTS / 'raw'}")
print(pd.Series([r["phase"] for r in REG]).value_counts().to_string())

In [ ]:
DET = {(r["L"], r["H"], r["strat"]): r for r in REG if r["phase"] == "detail"}
FIN = {(r["L"], r["H"]): r for r in REG if r["phase"] == "top5" and r.get("saved")}
print(f"Detalle por fold: {len(DET)} de 48 | modelos finales: {len(FIN)} de 16")

def fila(m, bds=None):
    return dict(MAPE=m["mape"], MAE=m["mae"], RMSE=m["rmse"], MSE=m["mse"], BDS_pvalue_h1=bds)

def tabla_folds(L, H, s):
    d = DET[(L, H, s)]
    t = pd.DataFrame([dict(Fold=f"Fold {f['fold']+1}", n_val=f["n_val"], **fila(f["keras_val"], f["bds_p_h1"]))
                      for f in d["folds"]]).set_index("Fold")
    num = t.drop(columns="n_val").astype(float)
    t.loc["Mean"] = num.mean(); t.loc["Std"] = num.std()
    return t

## 3.1 Métricas por fold (validación, USD)

Mostramos la configuración representativa (lag 14, H 14) con las tres estrategias. Las tablas de las 48 combinaciones quedan en `results/metricas_val_por_fold.csv`.

In [ ]:
largas = []
for (L, H, s), d in DET.items():
    for f in d["folds"]:
        largas.append(dict(estrategia=NOMBRE[s], lag=L, H=H, fold=f["fold"] + 1, n_train=f["n_train"],
                           n_val=f["n_val"], **fila(f["keras_val"], f["bds_p_h1"]),
                           RMSE_mlpregressor=f["mlpreg_val"]["rmse"], RMSE_ingenuo=f["naive_val"]["rmse"]))
LARGA = pd.DataFrame(largas).sort_values(["estrategia", "lag", "H", "fold"])
LARGA.to_csv(RESULTS / "metricas_val_por_fold.csv", index=False)
for s in ESTRATEGIAS:
    if (14, 14, s) in DET:
        print(f"\n{NOMBRE[s]} — lag 14, H 14")
        display(tabla_folds(14, 14, s).round(3))

Con Forward Chaining y K-Fold cada fold de `timeseries-cv` valida sobre una sola ventana, por lo que el test BDS por fold no es calculable (se reporta vacío); con Group K-Fold cada fold valida sobre decenas de ventanas.

## 3.2 Métricas por paso de horizonte

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3.6), sharey=False)
for j, H in enumerate(HS):
    for L in LAGS:
        if (L, H, "gk") in DET:
            rh = np.mean([f["keras_val"]["rmse_h"] for f in DET[(L, H, "gk")]["folds"]], axis=0)
            axs[j].plot(range(1, H + 1), rh, marker="o", ms=3, color=COL_LAG[L], label=f"lag {L}")
    axs[j].set_title(f"H = {H}"); axs[j].set_xlabel("Paso h (días)")
axs[0].set_ylabel("RMSE validación (USD)"); axs[-1].legend()
fig.suptitle("RMSE promedio por paso de horizonte (Group K-Fold)"); fig.tight_layout()
guardar(fig, "eval_rmse_por_paso"); plt.show()

## 3.3 Tabla consolidada lag × horizonte

In [ ]:
CONS = LARGA.groupby(["estrategia", "lag", "H"])[["RMSE", "MAPE", "MAE", "MSE", "BDS_pvalue_h1"]].mean().reset_index()
CONS.to_csv(RESULTS / "tabla_consolidada_lag_h.csv", index=False)
for met in ("RMSE", "MAPE"):
    print(f"\n{met} promedio de validación — Group K-Fold")
    display(CONS[CONS.estrategia == "Group K-Fold"].pivot(index="lag", columns="H", values=met).round(2))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, s in zip(axs, ESTRATEGIAS):
    g = CONS[CONS.estrategia == NOMBRE[s]]
    for L in LAGS:
        gl = g[g.lag == L].sort_values("H")
        ax.plot(gl.H, gl.RMSE, marker="o", ms=6, color=COL_LAG[L], label=f"lag {L}")
    ax.set_title(NOMBRE[s]); ax.set_xticks(HS); ax.set_xlabel("Horizonte H (días)")
axs[0].set_ylabel("RMSE validación (USD)"); axs[-1].legend()
fig.suptitle("Degradación del error con el horizonte"); fig.tight_layout()
guardar(fig, "eval_rmse_vs_H"); plt.show()

## 3.4 Comparación de estrategias de validación

In [ ]:
COMP = CONS.rename(columns={"estrategia": "estrategia"})[["estrategia", "lag", "H", "RMSE", "MAPE", "MAE", "MSE", "BDS_pvalue_h1"]]
COMP.to_csv(RESULTS / "comparacion_estrategias.csv", index=False)
COMP.pivot_table(index=["lag", "H"], columns="estrategia", values="RMSE").round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
w = 0.26
for k, s in enumerate(ESTRATEGIAS):
    if (14, 14, s) in DET:
        v = [f["keras_val"]["rmse"] for f in DET[(14, 14, s)]["folds"]]
        ax.bar(np.arange(len(v)) + (k - 1) * w, v, width=w - 0.02, color=COL_EST[s], label=NOMBRE[s])
ax.set_xticks(range(5)); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylabel("RMSE validación (USD)"); ax.set_title("RMSE por fold y estrategia (lag 14, H 14)"); ax.legend()
guardar(fig, "eval_rmse_por_fold_estrategias"); plt.show()

### Esquema de partición de cada estrategia

In [ ]:
def esquema(L=14, H=14):
    fig, axs = plt.subplots(3, 1, figsize=(11, 5.5), sharex=True)
    for ax, s in zip(axs, ESTRATEGIAS):
        z = np.load(RESULTS / "preds" / f"detail_L{L}_H{H}_{s}.npz")
        nf = len([k for k in z.files if k.endswith("_va_start")])
        for j in range(nf):
            for parte, c, lw in (("tr", COL[0], 5), ("va", COL[1], 9)):
                st = z[f"f{j}_{parte}_start"]; st = st[st >= 0] + L
                ax.scatter(DATES[st], np.full(len(st), j + 1), marker="|", s=lw * 12, color=c, lw=1)
        ax.axvspan(DATES[N_CV], DATES[-1], color="0.85", lw=0)
        ax.set_yticks(range(1, nf + 1)); ax.set_ylabel("Fold"); ax.set_title(NOMBRE[s], loc="left")
        ax.grid(False)
    from matplotlib.lines import Line2D
    axs[0].legend([Line2D([], [], color=COL[0], lw=3), Line2D([], [], color=COL[1], lw=3),
                   Line2D([], [], color="0.75", lw=6)], ["Train (inicio del objetivo)", "Validación", "Hold-out"],
                  ncol=3, loc="upper left", bbox_to_anchor=(0, 1.45))
    fig.tight_layout(); guardar(fig, f"esquema_particiones_L{L}_H{H}"); plt.show()

esquema()

## 3.5 Referencias: MLPRegressor y pronóstico ingenuo

In [ ]:
BASEL = LARGA.groupby(["estrategia", "lag", "H"])[["RMSE", "RMSE_mlpregressor", "RMSE_ingenuo"]].mean().reset_index()
BASEL["MLP_mejor_que_ingenuo"] = BASEL.RMSE < BASEL.RMSE_ingenuo
BASEL.to_csv(RESULTS / "baselines_val.csv", index=False)
print("Configuraciones donde el MLP Keras supera al ingenuo:",
      BASEL.groupby("estrategia").MLP_mejor_que_ingenuo.agg(["sum", "size"]).to_string())
BASEL[BASEL.estrategia == "Group K-Fold"].round(1)

## 3.6 Modelo final en el hold-out

In [ ]:
def ingenuo_holdout(L, H):
    idx = range(N_CV, len(y) - H + 1)
    P = np.array([[y[i - 1]] * H for i in idx]); Y = np.array([y[i:i + H] for i in idx])
    e = P - Y
    return float(np.sqrt((e ** 2).mean())), float(np.mean(np.abs(e / Y)) * 100)

filas = []
for (L, H), r in sorted(FIN.items()):
    rmse_i, mape_i = ingenuo_holdout(L, H)
    filas.append(dict(lag=L, H=H, arquitectura=str(tuple(r["arch"])), dropout_mask=str(tuple(r["mask"])),
                      RMSE=r["holdout_rmse"], MAPE=r["holdout_mape"], MAE=r["holdout_mae"], MSE=r["holdout_mse"],
                      RMSE_ingenuo=rmse_i, MAPE_ingenuo=mape_i, ventanas=r["n_holdout_windows"]))
HOLD = pd.DataFrame(filas)
HOLD.to_csv(RESULTS / "holdout_modelo_final.csv", index=False)
HOLD.round(2)

## 3.7 Series reales y predichas (train, validación y hold-out)

Para cada (lag, H) graficamos la predicción a un paso (h = 1) del mejor, el mediano y el peor fold de Group K-Fold según RMSE de validación, junto con el modelo final en el hold-out.

In [ ]:
def serie_config(L, H, s="gk", mostrar=True):
    d = DET[(L, H, s)]; z = np.load(RESULTS / "preds" / f"detail_L{L}_H{H}_{s}.npz")
    rm = np.array([f["keras_val"]["rmse"] for f in d["folds"]]); orden = np.argsort(rm)
    elegidos = {"mejor": orden[0], "mediano": orden[len(orden) // 2], "peor": orden[-1]}
    fh = RESULTS / "models" / f"own_L{L}_H{H}_preds.npz"
    fig, axs = plt.subplots(1, 3, figsize=(16, 3.4), sharey=True)
    for ax, (et, j) in zip(axs, elegidos.items()):
        for parte, cr, cp, nom in (("tr", "0.55", COL[0], "train"), ("va", "0.15", COL[1], "val")):
            st = z[f"f{j}_{parte}_start"]; ok = st >= 0; t = DATES[st[ok] + L]
            o = np.argsort(t)
            ax.plot(t[o], z[f"f{j}_{parte}_real"][ok][o, 0], ".", ms=2, color=cr, label=f"{nom} actual")
            ax.plot(t[o], z[f"f{j}_{parte}_pred"][ok][o, 0], ".", ms=2, color=cp, label=f"{nom} predicho")
        if fh.exists():
            hz = np.load(fh); th = DATES[N_CV:N_CV + len(hz["real"])]
            ax.plot(th, hz["real"][:, 0], color="0.3", lw=1, label="hold-out actual")
            ax.plot(th, hz["pred"][:, 0], color=COL[2], lw=1.2, label="hold-out predicho")
        ax.set_yscale("log"); ax.set_title(f"lag {L}, H {H} — fold {et} ({j+1}), RMSE val {rm[j]:,.0f} USD", fontsize=9)
    axs[0].set_ylabel("USD (log)")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, ncol=6, fontsize=8, markerscale=4, loc="lower center", bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(); guardar(fig, f"serie_L{L}_H{H}_{s}")
    plt.show() if mostrar else plt.close(fig)

for L in LAGS:
    for H in HS:
        if (L, H, "gk") in DET:
            serie_config(L, H)